# Gateway User Attribution — "Who Called What" at the MCP Gateway

When several teams and agents share one Amazon Bedrock AgentCore **Gateway**,
a natural governance question follows: *which user actually invoked which tool,
with which arguments, and what came back?* Answering it per MCP server means
every server re-implements identity handling and logging. This sample answers it
**once, centrally at the gateway** — with **zero instrumentation in the MCP
servers** — using a gateway **RESPONSE interceptor**.

A RESPONSE interceptor runs after the target produces a result but before the
gateway returns it, and for an MCP target its input carries **both** the original
request (including the validated JWT) **and** the response body. So one small
Lambda can emit a single record tying together
`user → tool → arguments → response`, landing in one log group with no
cross-service plumbing.

### Why it matters

- **Governance / audit:** a single, authoritative record of who did what through
  the gateway, independent of how each MCP server is written.
- **Lead-generation signal:** a whitepaper *download* used to be the marketing
  signal. Now a user asking an agent `GetDocument(whitepaper-123)` is the same
  signal — captured at the gateway, ready to pipe onward via a CloudWatch Logs
  subscription filter → Firehose → your marketing platform.

### What you'll learn

- How to attach a **RESPONSE interceptor** to an AgentCore Gateway
  (`passRequestHeaders=true`) and read the validated JWT claims from it.
- How to capture `user → tool → arguments → response` for every call with no
  code in the MCP servers.
- How to surface it on a CloudWatch dashboard and query it with Logs Insights.

### Tutorial details

| Information | Details |
|:--|:--|
| Tutorial type | Interactive, self-contained notebook |
| Gateway inbound auth | Amazon Cognito (CUSTOM_JWT) |
| Interceptor | RESPONSE, Lambda, `passRequestHeaders=true` |
| MCP targets | Two Lambda targets (document + analytics tools) |
| State | None (mock in-memory tool data) |
| Observability | CloudWatch metrics, vended logs, dashboard, Logs Insights |
| Example complexity | Intermediate |
| Clean up | Yes — the final section deletes every resource created |

> This notebook is **self-contained and idempotent** (a random suffix keeps every
> resource name unique per run). Run it top to bottom; the **Cleanup** section at
> the end removes everything it created.

## Architecture

```mermaid
flowchart TB
    U(["User / Agent"]) -->|"login → access token (sub, username, client_id)"| GW
    subgraph GWBOX["AgentCore Gateway — CUSTOM_JWT inbound auth"]
        GW["Gateway\nvalidates the JWT"]
        INT["RESPONSE interceptor (Lambda)\npassRequestHeaders=true\nsees request JWT + response body\nemits { user → tool → arguments → response }"]
        T1["document-server target\nGetDocument / ListDocuments / SearchDocuments"]
        T2["analytics-server target\nGetMetrics / GenerateReport"]
        GW --> T1
        GW --> T2
        T1 --> INT
        T2 --> INT
    end
    INT --> CW
    GW --> CW
    subgraph CW["CloudWatch"]
        M["AWS/Bedrock-AgentCore metrics\n(automatic: volume, latency, errors)"]
        L["Interceptor log group\nuser → tool → arguments → response"]
        D["Dashboard + Logs Insights"]
    end
```

The flow: the user authenticates with Cognito and calls the gateway with the
resulting JWT. The gateway validates it, invokes the tool target, then — on the
way back — hands the request (with the JWT) **and** the response to the RESPONSE
interceptor, which logs one attribution record. Everything lands in CloudWatch.

## 1. Prerequisites & configuration

- An AWS account with Amazon Bedrock AgentCore available in your region.
- Credentials permitting `bedrock-agentcore-control`, `lambda`, `iam`,
  `cognito-idp`, `logs`, `cloudwatch`, `sts`.
- `pip install -r requirements.txt`

In [ ]:
import base64, io, json, secrets, time, zipfile, uuid
import boto3
from botocore.exceptions import ClientError

# ---- Configuration -------------------------------------------------------
REGION = "us-east-1"
# A short random suffix keeps every named resource unique per run so repeated
# runs (or multiple people in one account) never collide.
SUFFIX = uuid.uuid4().hex[:8]

PREFIX              = f"gw-attr-{SUFFIX}"
USER_POOL_NAME      = f"{PREFIX}-pool"
DEMO_USERNAME       = "demo-user"
DEMO_USER_SECRET    = secrets.token_urlsafe(16) + "Aa1!"   # random per run; satisfies the pool password policy
GATEWAY_NAME        = f"{PREFIX}-gateway"
GATEWAY_ROLE_NAME   = f"{PREFIX}-gw-role"
DOC_FN_NAME         = f"{PREFIX}-document-server"
ANALYTICS_FN_NAME   = f"{PREFIX}-analytics-server"
INTERCEPTOR_FN_NAME = f"{PREFIX}-user-attribution-interceptor"
DASHBOARD_NAME      = f"{PREFIX}-dashboard"

session   = boto3.Session(region_name=REGION)
sts       = session.client("sts")
iam       = session.client("iam")
lam       = session.client("lambda")
cognito   = session.client("cognito-idp")
logs      = session.client("logs")
cw        = session.client("cloudwatch")
agc       = session.client("bedrock-agentcore-control")

ACCOUNT = sts.get_caller_identity()["Account"]
print(f"Account {ACCOUNT}, region {REGION}, resource prefix '{PREFIX}'")

# A dict we accumulate created-resource identifiers into, so Cleanup can find
# everything even if a later cell fails.
state = {}

## 2. Cognito user pool + demo user

The gateway trusts this user pool as its JWT issuer. We create one public app
client (no secret) and a `demo-user` whose access token the gateway will
validate on every tool call.

In [ ]:
pool = cognito.create_user_pool(
    PoolName=USER_POOL_NAME,
    Policies={"PasswordPolicy": {"MinimumLength": 8, "RequireUppercase": True,
        "RequireLowercase": True, "RequireNumbers": True, "RequireSymbols": True}},
    AutoVerifiedAttributes=["email"],
)
USER_POOL_ID = pool["UserPool"]["Id"]
state["user_pool_id"] = USER_POOL_ID

client = cognito.create_user_pool_client(
    UserPoolId=USER_POOL_ID,
    ClientName=f"{PREFIX}-client",
    GenerateSecret=False,
    ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
)
CLIENT_ID = client["UserPoolClient"]["ClientId"]
state["client_id"] = CLIENT_ID

ISSUER = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}"
DISCOVERY_URL = f"{ISSUER}/.well-known/openid-configuration"

# Create demo-user and set a permanent password.
cognito.admin_create_user(UserPoolId=USER_POOL_ID, Username=DEMO_USERNAME,
    UserAttributes=[{"Name": "email", "Value": "demo-user@example.com"},
                    {"Name": "email_verified", "Value": "true"}],
    MessageAction="SUPPRESS")
cognito.admin_set_user_password(UserPoolId=USER_POOL_ID, Username=DEMO_USERNAME,
    Password=DEMO_USER_SECRET, Permanent=True)

print("User pool :", USER_POOL_ID)
print("Client id :", CLIENT_ID)
print("Discovery :", DISCOVERY_URL)

## 3. Deploy the Lambdas

Two MCP tool targets (`document_server`, `analytics_server`) and the
user-attribution `interceptor`. Each is zipped from the `lambdas/` folder next
to this notebook and deployed with a basic execution role.

In [ ]:
LAMBDA_DIR = "lambdas"

def _zip_single(path):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="index.py")
    buf.seek(0)
    return buf.read()

# A minimal Lambda execution role (CloudWatch Logs only).
assume = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
    "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
lrole = iam.create_role(RoleName=f"{PREFIX}-lambda-role",
    AssumeRolePolicyDocument=json.dumps(assume),
    Description="Execution role for gateway user-attribution sample Lambdas")
LAMBDA_ROLE_ARN = lrole["Role"]["Arn"]
state["lambda_role"] = f"{PREFIX}-lambda-role"
iam.attach_role_policy(RoleName=f"{PREFIX}-lambda-role",
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
print("Waiting for IAM role to propagate..."); time.sleep(10)

def deploy_fn(name, src_file):
    code_zip = _zip_single(f"{LAMBDA_DIR}/{src_file}")
    fn = lam.create_function(FunctionName=name, Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN, Handler="index.lambda_handler",
        Code={"ZipFile": code_zip}, Timeout=30, MemorySize=128)
    waiter = lam.get_waiter("function_active_v2")
    waiter.wait(FunctionName=name)
    print("  deployed", name)
    return fn["FunctionArn"]

DOC_ARN         = deploy_fn(DOC_FN_NAME, "document_server.py")
ANALYTICS_ARN   = deploy_fn(ANALYTICS_FN_NAME, "analytics_server.py")
INTERCEPTOR_ARN = deploy_fn(INTERCEPTOR_FN_NAME, "interceptor.py")
state.update(doc_fn=DOC_FN_NAME, analytics_fn=ANALYTICS_FN_NAME, interceptor_fn=INTERCEPTOR_FN_NAME)
INTERCEPTOR_LOG_GROUP = f"/aws/lambda/{INTERCEPTOR_FN_NAME}"

## 4. IAM role for the gateway + the gateway itself

The gateway assumes a role trusting `bedrock-agentcore.amazonaws.com` and
granting `lambda:InvokeFunction` on the three Lambdas (two tool targets plus
the interceptor). Inbound auth is **CUSTOM_JWT** validated against the Cognito
pool, so every request must carry a valid access token.

In [ ]:
gw_assume = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
    "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
    "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT},
        "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT}:gateway/*"}}}]}
grole = iam.create_role(RoleName=GATEWAY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(gw_assume),
    Description="Assumed by AgentCore Gateway to invoke MCP target + interceptor Lambdas")
GATEWAY_ROLE_ARN = grole["Role"]["Arn"]
state["gateway_role"] = GATEWAY_ROLE_NAME
iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="invoke-targets",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{
        "Effect": "Allow", "Action": "lambda:InvokeFunction",
        "Resource": [DOC_ARN, ANALYTICS_ARN, INTERCEPTOR_ARN]}]}))
print("Waiting for gateway role to propagate..."); time.sleep(10)

def wait_gateway_ready(gid, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        gw = agc.get_gateway(gatewayIdentifier=gid)
        s = gw.get("status")
        if s == "READY": return gw
        if s and s.endswith("FAILED"): raise RuntimeError(f"{s}: {gw.get('statusReasons')}")
        print("  gateway status", s, "- waiting"); time.sleep(5)
    raise TimeoutError("gateway not READY")

gw = agc.create_gateway(name=GATEWAY_NAME, roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP", authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": DISCOVERY_URL, "allowedClients": [CLIENT_ID]}},
    exceptionLevel="DEBUG",
    description="User-attribution sample gateway (CUSTOM_JWT, Cognito)")
GATEWAY_ID = gw["gatewayId"]
state["gateway_id"] = GATEWAY_ID
gw = wait_gateway_ready(GATEWAY_ID)
GATEWAY_ARN = gw["gatewayArn"]
GATEWAY_URL = gw["gatewayUrl"]
print("Gateway", GATEWAY_ID, "READY")
print("MCP endpoint:", GATEWAY_URL)

## 5. Attach the two Lambda tool targets

Each target declares its tool schema inline. Tool names are exposed through MCP
prefixed with the target name, e.g. `document-server___GetDocument`.

In [ ]:
DOCUMENT_TOOLS = [
  {"name": "GetDocument", "description": "Retrieve a single document by its ID.",
   "inputSchema": {"type": "object", "properties": {"documentId": {"type": "string",
     "description": "The document ID, e.g. 'whitepaper-123'."}}, "required": ["documentId"]}},
  {"name": "ListDocuments", "description": "List documents, optionally by category.",
   "inputSchema": {"type": "object", "properties": {"category": {"type": "string"}}, "required": []}},
  {"name": "SearchDocuments", "description": "Keyword search over documents.",
   "inputSchema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
]
ANALYTICS_TOOLS = [
  {"name": "GetMetrics", "description": "Get engagement metrics for a product.",
   "inputSchema": {"type": "object", "properties": {"product": {"type": "string",
     "description": "Product key, e.g. 'edge-sensor'."}, "period": {"type": "string"}}, "required": ["product"]}},
  {"name": "GenerateReport", "description": "Generate a short summary report.",
   "inputSchema": {"type": "object", "properties": {"type": {"type": "string"}}, "required": ["type"]}},
]

def add_target(name, arn, tools):
    agc.create_gateway_target(gatewayIdentifier=GATEWAY_ID, name=name,
        targetConfiguration={"mcp": {"lambda": {"lambdaArn": arn,
            "toolSchema": {"inlinePayload": tools}}}},
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}])
    print("  target added:", name)

add_target("document-server", DOC_ARN, DOCUMENT_TOOLS)
add_target("analytics-server", ANALYTICS_ARN, ANALYTICS_TOOLS)
wait_gateway_ready(GATEWAY_ID)
print("Targets attached.")

## 6. Attach the RESPONSE interceptor

This is the piece that makes attribution possible. `UpdateGateway` replaces the
whole config, so we re-supply the create-time fields and add
`interceptorConfigurations` at the **RESPONSE** point with `passRequestHeaders=true`, so the interceptor
receives the `Authorization` JWT.

> `GetGateway` does **not** echo `interceptorConfigurations` back, so verify the
> interceptor is live by making a tool call and checking its log group — not by
> reading the gateway config.

In [ ]:
gw = agc.get_gateway(gatewayIdentifier=GATEWAY_ID)
grant_sid = "AllowGatewayInvokeInterceptor"
try:
    lam.add_permission(FunctionName=INTERCEPTOR_FN_NAME, StatementId=grant_sid,
        Action="lambda:InvokeFunction", Principal="bedrock-agentcore.amazonaws.com",
        SourceArn=GATEWAY_ARN)
except ClientError as e:
    if e.response["Error"]["Code"] != "ResourceConflictException": raise

update_args = {"gatewayIdentifier": GATEWAY_ID, "name": gw["name"],
    "roleArn": gw["roleArn"], "protocolType": gw["protocolType"],
    "authorizerType": gw["authorizerType"],
    "authorizerConfiguration": gw["authorizerConfiguration"],
    "interceptorConfigurations": [{
        "interceptor": {"lambda": {"arn": INTERCEPTOR_ARN}},
        "interceptionPoints": ["RESPONSE"],
        "inputConfiguration": {"passRequestHeaders": True}}]}
for f in ("exceptionLevel", "description"):
    if gw.get(f): update_args[f] = gw[f]

agc.update_gateway(**update_args)
wait_gateway_ready(GATEWAY_ID)
print("RESPONSE interceptor attached (passRequestHeaders=true).")

## 7. Enable gateway vended logging

Deliver the gateway's request/response bodies to a CloudWatch log group so the
dashboard and queries can show *what was asked and what came back*.

In [ ]:
VENDED_LOG_GROUP = f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{GATEWAY_ID}"
try:
    logs.create_log_group(logGroupName=VENDED_LOG_GROUP)
except logs.exceptions.ResourceAlreadyExistsException:
    pass
logs.put_retention_policy(logGroupName=VENDED_LOG_GROUP, retentionInDays=7)
log_group_arn = f"arn:aws:logs:{REGION}:{ACCOUNT}:log-group:{VENDED_LOG_GROUP}:*"

src_name  = f"{PREFIX}-src"[:60]
dest_name = f"{PREFIX}-dest"[:60]
logs.put_delivery_source(name=src_name, resourceArn=GATEWAY_ARN, logType="APPLICATION_LOGS")
dest = logs.put_delivery_destination(name=dest_name,
    deliveryDestinationConfiguration={"destinationResourceArn": log_group_arn})
logs.create_delivery(deliverySourceName=src_name,
    deliveryDestinationArn=dest["deliveryDestination"]["arn"])
state.update(vended_log_group=VENDED_LOG_GROUP, delivery_source=src_name, delivery_dest=dest_name)
print("Vended logging active:", VENDED_LOG_GROUP)

## 8. Create the CloudWatch dashboard

Metrics in `AWS/Bedrock-AgentCore` are emitted automatically by the gateway.
The two Logs Insights tables — **who called what** and **request/response
bodies** — are the distinctive part of this sample.

In [ ]:
ns = "AWS/Bedrock-AgentCore"
def method_metric(metric, method, stat, label):
    return [ns, metric, "Resource", GATEWAY_ARN, "Operation", "InvokeGateway",
            "Protocol", "MCP", "Method", method, {"stat": stat, "label": label}]

body = {"widgets": [
  {"type": "text", "width": 24, "height": 2, "properties": {"markdown":
     "# Gateway User Attribution\nEvery MCP tool call captured centrally - with zero code in the MCP servers."}},
  {"type": "metric", "width": 12, "height": 7, "properties": {"title": "tools/call volume (per tool)",
     "region": REGION, "stacked": True, "metrics": [[{"expression":
       f"SEARCH('{{{ns},Method,Name,Operation,Protocol,Resource}} Resource=\"{GATEWAY_ARN}\" MetricName=\"Invocations\" Method=\"tools/call\"', 'Sum', 60)", "label": ""}]]}},
  {"type": "metric", "width": 12, "height": 7, "properties": {"title": "tools/call latency (ms)",
     "region": REGION, "metrics": [method_metric("Latency", "tools/call", "p50", "p50"),
       method_metric("Latency", "tools/call", "p90", "p90"), method_metric("Latency", "tools/call", "p99", "p99")]}},
  {"type": "log", "width": 24, "height": 12, "properties": {"title": "Who called what - user -> tool -> arguments -> response",
     "region": REGION, "view": "table", "query":
       f"SOURCE '{INTERCEPTOR_LOG_GROUP}' | fields @timestamp, `user.username`, `user.sub`, tool, target, `arguments.product`, `arguments.documentId`, `arguments.query`, `arguments.type`, status_code, response | filter signal = 'mcp.tool_call' | sort @timestamp desc | limit 50"}},
]}
cw.put_dashboard(DashboardName=DASHBOARD_NAME, DashboardBody=json.dumps(body))
state["dashboard"] = DASHBOARD_NAME
print("Dashboard created:", DASHBOARD_NAME)
print(f"Console: https://console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards/dashboard/{DASHBOARD_NAME}")

## 9. Generate traffic as `demo-user`

Authenticate with Cognito to get an access token, then call the gateway's MCP
endpoint directly over HTTP (JSON-RPC). Each `tools/call` carries the user's
JWT in the `Authorization` header — exactly what the interceptor reads.

In [ ]:
import requests

auth = cognito.initiate_auth(ClientId=CLIENT_ID, AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": DEMO_USERNAME, "PASSWORD": DEMO_USER_SECRET})
ACCESS_TOKEN = auth["AuthenticationResult"]["AccessToken"]
print("Got access token for", DEMO_USERNAME)

def mcp_call(method, params=None, _id=[0]):
    _id[0] += 1
    r = requests.post(GATEWAY_URL, headers={
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}},
        timeout=30)
    return r

def tool_call(name, arguments):
    r = mcp_call("tools/call", {"name": name, "arguments": arguments})
    print(f"  {name}({arguments}) -> HTTP {r.status_code}")
    return r

# Discover tools, then make a handful of attributable calls.
mcp_call("initialize", {"protocolVersion": "2024-11-05", "capabilities": {},
    "clientInfo": {"name": "sample", "version": "1.0"}})
mcp_call("tools/list")

# Generate ~20 varied calls spread over ~3 minutes so the dashboard's
# volume/latency graphs show a trend rather than a single spike.
import random
CALLS = [
    ("document-server___ListDocuments", {}),
    ("document-server___GetDocument", {"documentId": "whitepaper-123"}),
    ("document-server___GetDocument", {"documentId": "datasheet-edge-sensor"}),
    ("document-server___GetDocument", {"documentId": "appnote-gateway-router"}),
    ("document-server___SearchDocuments", {"query": "edge"}),
    ("document-server___SearchDocuments", {"query": "gateway"}),
    ("analytics-server___GetMetrics", {"product": "edge-sensor"}),
    ("analytics-server___GetMetrics", {"product": "gateway-router"}),
    ("analytics-server___GetMetrics", {"product": "secure-controller"}),
    ("analytics-server___GenerateReport", {"type": "weekly-summary"}),
]
N_CALLS = 20
WINDOW_SECS = 180
for i in range(N_CALLS):
    name, args = random.choice(CALLS)
    tool_call(name, args)
    if i < N_CALLS - 1:
        time.sleep(WINDOW_SECS / (N_CALLS - 1))  # spread evenly across ~3 min
print(f"Traffic generated: {N_CALLS} calls over ~{WINDOW_SECS}s.")

## 10. Inspect — who called what

Query the interceptor's log group with CloudWatch Logs Insights. Each row maps
a Cognito user to the tool and arguments — identity extracted centrally at the
gateway, never handled by the MCP servers. (Logs take ~30–60s to appear; the
cell retries.)

In [ ]:
query = ("fields @timestamp, `user.username`, `user.sub`, tool, target, "
         "`arguments.product`, `arguments.documentId`, `arguments.query`, "
         "status_code, response "
         "| filter signal = 'mcp.tool_call' | sort @timestamp desc | limit 50")

def run_insights(log_group, q, expected=1, waves=18, wave_secs=10):
    """Poll Logs Insights until at least `expected` rows appear or we run out
    of waves. CloudWatch indexes records with a delay, and the traffic cell
    spreads calls over ~3 min, so we wait for the full set before displaying."""
    best = []
    for attempt in range(waves):
        start = int(time.time()) - 1800
        qid = logs.start_query(logGroupName=log_group, startTime=start,
            endTime=int(time.time()), queryString=q)["queryId"]
        while True:
            res = logs.get_query_results(queryId=qid)
            if res["status"] in ("Complete", "Failed", "Cancelled"): break
            time.sleep(1)
        rows = res.get("results", [])
        if len(rows) > len(best): best = rows
        if len(best) >= expected: return best
        print(f"  have {len(best)}/{expected} records (attempt {attempt+1}/{waves}), waiting for CloudWatch to index...")
        time.sleep(wave_secs)
    return best

# N_CALLS comes from the traffic cell. CloudWatch may index one or two
# fewer records than calls made (a non-tools/call message, or a collapsed
# duplicate), so wait for most of them rather than hanging on an exact match.
rows = run_insights(INTERCEPTOR_LOG_GROUP, query, expected=max(1, N_CALLS - 2))
print(f"{len(rows)} attribution record(s) retrieved. See the next cell for the table and chart.")

## 10b. Demo view — attribution table + dashboard link

A single cell for the live demo: it prints the captured
`user → tool → arguments → response` records as a compact table, then the
CloudWatch dashboard link so you can show both the raw records and the
rendered dashboard.

In [ ]:
# Render the attribution records (from the previous cell) as a table.
def _cell(row, name):
    for f in row:
        if f['field'] == name:
            return f['value']
    return ''

cols = [('user.username', 'user'), ('tool', 'tool'), ('target', 'target'),
        ('arguments.product', 'product'), ('arguments.documentId', 'documentId'),
        ('arguments.query', 'query'), ('status_code', 'status'), ('response', 'response')]
widths = {h: len(h) for _, h in cols}
table = []
for row in rows:
    r = {h: (_cell(row, f) or '') for f, h in cols}
    for h in widths:
        widths[h] = max(widths[h], min(len(r[h]), 40))
    table.append(r)

def fmt(v, w):
    v = v if len(v) <= w else v[: w - 1] + '…'
    return v.ljust(w)

header = '  '.join(fmt(h, widths[h]) for _, h in cols)
print(header)
print('-' * len(header))
for r in table:
    print('  '.join(fmt(r[h], widths[h]) for _, h in cols))

print(f'\n{len(table)} record(s).')
print(f'\nCloudWatch dashboard: '
      f'https://console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards/dashboard/{DASHBOARD_NAME}')

## 10c. Tool-call volume per tool

A quick visual of how many calls each tool received, built from the same
attribution records. (The CloudWatch dashboard shows this live too; this cell
gives an inline view without leaving the notebook.)

In [ ]:
import collections
import matplotlib.pyplot as plt

counts = collections.Counter(_cell(r, 'tool') for r in rows if _cell(r, 'tool'))
if counts:
    tools = sorted(counts, key=counts.get)
    vals = [counts[t] for t in tools]
    plt.figure(figsize=(7, 0.5 * len(tools) + 1))
    plt.barh(tools, vals, color='#2563eb')
    plt.xlabel('calls')
    plt.title(f'Tool-call volume per tool ({sum(vals)} calls by demo-user)')
    for i, v in enumerate(vals):
        plt.text(v, i, f' {v}', va='center')
    plt.tight_layout()
    plt.show()
else:
    print('No records to chart yet - re-run the inspect cell once CloudWatch has indexed the calls.')

## The lead-generation tie-in

`demo-user` asking `GetDocument(whitepaper-123)` is the direct equivalent of a
whitepaper-download event — same **who + what**, captured centrally at the
gateway with no per-server code. Pipe it onward with a CloudWatch Logs
subscription filter:

```
CloudWatch Logs  →  Subscription Filter  →  Firehose  →  marketing platform / S3 / lake
```

## 11. Cleanup

Deletes **every** resource this notebook created. Each step is wrapped so one
failure does not stop the rest. Run this when you are done to avoid any ongoing
CloudWatch charges.

In [ ]:
def safe(desc, fn):
    try:
        fn(); print("  deleted", desc)
    except Exception as e:
        print("  skip", desc, "-", type(e).__name__)

def retry(desc, fn, tries=6, delay=5):
    """Retry a delete that fails only because a dependent resource is still
    draining (async deletes raise ValidationException/ConflictException until
    the dependency is gone)."""
    for attempt in range(tries):
        try:
            fn(); print("  deleted", desc); return
        except Exception as e:
            name = type(e).__name__
            if name in ("ValidationException", "ConflictException", "ResourceInUseException") and attempt < tries - 1:
                print(f"  {desc}: {name}, waiting for dependency to drain ({attempt+1}/{tries})...")
                time.sleep(delay); continue
            print("  skip", desc, "-", name); return

# 1. Delete gateway targets, then WAIT for them to be gone before the gateway
#    (delete_gateway raises ValidationException while any target is attached).
if state.get("gateway_id"):
    try:
        for page in agc.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=state["gateway_id"]):
            for t in page.get("items", []):
                safe(f"target {t['name']}", lambda t=t: agc.delete_gateway_target(
                    gatewayIdentifier=state['gateway_id'], targetId=t['targetId']))
    except Exception as e:
        print("  (no targets to list)", type(e).__name__)
    # Poll until the target list is empty.
    for _ in range(12):
        remaining = []
        try:
            for page in agc.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=state["gateway_id"]):
                remaining += page.get("items", [])
        except Exception:
            break
        if not remaining: break
        print(f"  waiting for {len(remaining)} target(s) to drain..."); time.sleep(5)
    retry("gateway", lambda: agc.delete_gateway(gatewayIdentifier=state["gateway_id"]))
    time.sleep(5)

# 2. Log delivery: delete every delivery that references this source (there
#    can be more than one), confirm none remain, THEN delete the source and
#    destination (both raise ConflictException while any delivery links them).
#    describe_deliveries is PAGINATED - an account with many deliveries can
#    hide the matching one on a later page, so always paginate.
if state.get("delivery_source"):
    def _linked_deliveries():
        found = []
        for page in logs.get_paginator("describe_deliveries").paginate():
            for d in page.get("deliveries", []):
                if d.get("deliverySourceName") == state["delivery_source"]:
                    found.append(d)
        return found
    for attempt in range(8):
        linked = _linked_deliveries()
        if not linked:
            if attempt == 0: print("  no delivery linked to source")
            break
        for d in linked:
            safe(f"delivery {d['id']}", lambda d=d: logs.delete_delivery(id=d["id"]))
        time.sleep(4)
    retry("delivery source", lambda: logs.delete_delivery_source(name=state["delivery_source"]))
if state.get("delivery_dest"):
    retry("delivery destination", lambda: logs.delete_delivery_destination(name=state["delivery_dest"]))
if state.get("vended_log_group"):
    safe("vended log group", lambda: logs.delete_log_group(logGroupName=state["vended_log_group"]))

# Dashboard
if state.get("dashboard"):
    safe("dashboard", lambda: cw.delete_dashboards(DashboardNames=[state["dashboard"]]))

# Lambdas + their log groups
for key in ("doc_fn", "analytics_fn", "interceptor_fn"):
    name = state.get(key)
    if name:
        safe(f"lambda {name}", lambda name=name: lam.delete_function(FunctionName=name))
        safe(f"log group {name}", lambda name=name: logs.delete_log_group(logGroupName=f"/aws/lambda/{name}"))

# IAM roles (detach/delete inline + managed policies first)
def del_role(role):
    for p in iam.list_attached_role_policies(RoleName=role).get("AttachedPolicies", []):
        iam.detach_role_policy(RoleName=role, PolicyArn=p["PolicyArn"])
    for pn in iam.list_role_policies(RoleName=role).get("PolicyNames", []):
        iam.delete_role_policy(RoleName=role, PolicyName=pn)
    iam.delete_role(RoleName=role)
for key in ("gateway_role", "lambda_role"):
    if state.get(key):
        safe(f"role {state[key]}", lambda r=state[key]: del_role(r))

# Cognito user pool (removes the client + user with it)
if state.get("user_pool_id"):
    safe("user pool", lambda: cognito.delete_user_pool(UserPoolId=state["user_pool_id"]))

print("\nCleanup complete.")